# Revenue by Product - Business Metrics
Analyzes product performance across the entire catalog. Answers questions like:
- Which products generate the most revenue?
- What's the average order value per product?
- How many unique customers buy each product?
- When was each product first/last sold?

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.agg_revenue_by_product AS
SELECT 
  pr.product_number,
  pr.product_name,
  pr.category,
  pr.subcategory,
  
  -- Revenue metrics
  SUM(fs.sales_amount) AS total_revenue,
  SUM(fs.quantity) AS total_quantity,
  COUNT(DISTINCT fs.order_number) AS order_count,
  COUNT(DISTINCT fs.customer_key) AS unique_customers,
  
  -- Average metrics
  ROUND(AVG(fs.price), 2) AS avg_price,
  ROUND(AVG(fs.sales_amount), 2) AS avg_order_value,
  
  -- Time metrics
  MIN(fs.order_date) AS first_sale_date,
  MAX(fs.order_date) AS last_sale_date,
  DATEDIFF(MAX(fs.order_date), MIN(fs.order_date)) AS days_on_market,
  
  -- Rankings
  RANK() OVER (ORDER BY SUM(fs.sales_amount) DESC) AS revenue_rank

FROM workspace.gold.fact_sales fs
INNER JOIN workspace.gold.dim_products pr 
  ON fs.product_key = pr.product_key
  AND pr.is_current = TRUE

WHERE fs.order_date IS NOT NULL

GROUP BY pr.product_number, pr.product_name, 
         pr.category, pr.subcategory

In [0]:
%sql
SELECT 
  product_name,
  category,
  total_revenue,
  total_quantity,
  unique_customers,
  avg_order_value,
  revenue_rank
FROM workspace.gold.agg_revenue_by_product
ORDER BY total_revenue DESC
LIMIT 10

In [0]:
%sql
SELECT 
  COUNT(*) AS total_products,
  SUM(total_revenue) AS total_revenue_all_products,
  ROUND(AVG(total_revenue), 2) AS avg_revenue_per_product,
  MAX(total_revenue) AS top_product_revenue,
  MIN(total_revenue) AS lowest_product_revenue
FROM workspace.gold.agg_revenue_by_product